In [0]:
# dim date table

silver_workforce = (spark.table("silver_workforce_fte"))
silver_workforce.show(10)

silver_sickness = (spark.table("silver_sick_leave"))
silver_sickness.show(10)



In [0]:

from pyspark.sql.functions import col, lower,trim, row_number, window
from pyspark.sql.window import Window


workforce_date = (silver_workforce
    .select(col('month').alias("date"))
    .distinct()  
)

workforce_date.show()


In [0]:

sickness_date = (silver_sickness
    .select(col('month').alias("date"))
    .distinct()
  
)

sickness_date.show()

In [0]:
from pyspark.sql.functions import col, rank
from pyspark.sql.window import Window

window_spec = Window.orderBy("date")

all_dates = (
    workforce_date
    .unionByName(sickness_date)
    .select(col("date").alias("date"))
    .distinct()
)


dim_time_gold = (
    all_dates
    .withColumn(
        "date_key",
        rank().over(window_spec)
    )
    .select(
        "date_key",
        "date"
    )
)

display(dim_time_gold)

In [0]:
(dim_time_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/dim_date/"
    ) \
    .saveAsTable(
        "date_dimension"
    ))